In [1]:
import pandas as pd
import numpy as np
import joblib
import unittest
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split

# ---------------------------------------------------
# Helper Functions (same logic as feature engineering)
# ---------------------------------------------------

def tenure_group(t):
    if t <= 12:   return 0
    elif t <= 24: return 1
    elif t <= 48: return 2
    else:         return 3

def fix_total_charges(df):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['MonthlyCharges'], inplace=True)
    return df

def compute_support_score(row):
    return row['OnlineSecurity'] + row['TechSupport'] + \
           row['OnlineBackup'] + row['DeviceProtection']

def encode_churn(series):
    return series.map({'Yes': 1, 'No': 0})

# ---------------------------------------------------
# Test Suite
# ---------------------------------------------------

class TestTenureGroup(unittest.TestCase):

    def test_new_customer(self):
        assert tenure_group(1)  == 0
        assert tenure_group(12) == 0

    def test_developing_customer(self):
        assert tenure_group(13) == 1
        assert tenure_group(24) == 1

    def test_established_customer(self):
        assert tenure_group(25) == 2
        assert tenure_group(48) == 2

    def test_loyal_customer(self):
        assert tenure_group(49) == 3
        assert tenure_group(72) == 3

    def test_boundary_zero(self):
        assert tenure_group(0) == 0


class TestTotalChargesFix(unittest.TestCase):

    def test_space_replaced_by_monthly(self):
        df = pd.DataFrame({
            'TotalCharges': [' ', '200.0', '350.5'],
            'MonthlyCharges': [50.0, 200.0, 350.5]
        })
        result = fix_total_charges(df)
        assert result['TotalCharges'].isnull().sum() == 0
        assert result['TotalCharges'].iloc[0] == 50.0

    def test_valid_values_unchanged(self):
        df = pd.DataFrame({
            'TotalCharges': ['100.0', '200.0'],
            'MonthlyCharges': [50.0, 80.0]
        })
        result = fix_total_charges(df)
        assert result['TotalCharges'].iloc[0] == 100.0
        assert result['TotalCharges'].iloc[1] == 200.0

    def test_no_nulls_after_fix(self):
        df = pd.DataFrame({
            'TotalCharges': [' ', ' ', '300.0'],
            'MonthlyCharges': [60.0, 70.0, 300.0]
        })
        result = fix_total_charges(df)
        assert result['TotalCharges'].isnull().sum() == 0


class TestSupportScore(unittest.TestCase):

    def test_all_services_active(self):
        row = pd.Series({'OnlineSecurity': 1, 'TechSupport': 1,
                         'OnlineBackup': 1, 'DeviceProtection': 1})
        assert compute_support_score(row) == 4

    def test_no_services(self):
        row = pd.Series({'OnlineSecurity': 0, 'TechSupport': 0,
                         'OnlineBackup': 0, 'DeviceProtection': 0})
        assert compute_support_score(row) == 0

    def test_partial_services(self):
        row = pd.Series({'OnlineSecurity': 1, 'TechSupport': 0,
                         'OnlineBackup': 1, 'DeviceProtection': 0})
        assert compute_support_score(row) == 2

    def test_score_range(self):
        for score in range(5):
            assert 0 <= score <= 4


class TestChurnEncoding(unittest.TestCase):

    def test_yes_maps_to_one(self):
        s = pd.Series(['Yes', 'No', 'Yes'])
        result = encode_churn(s)
        assert result.iloc[0] == 1
        assert result.iloc[2] == 1

    def test_no_maps_to_zero(self):
        s = pd.Series(['No', 'No'])
        result = encode_churn(s)
        assert result.iloc[0] == 0

    def test_sum_correct(self):
        s = pd.Series(['Yes', 'No', 'Yes', 'No', 'Yes'])
        result = encode_churn(s)
        assert result.sum() == 3


class TestModelOutput(unittest.TestCase):

    def test_model_loads(self):
        model = joblib.load('../outputs/model.pkl')
        assert model is not None

    def test_prediction_shape(self):
        model = joblib.load('../outputs/model.pkl')
        df = pd.read_csv('../data/processed/features.csv')
        bool_cols = df.select_dtypes(include='bool').columns
        df[bool_cols] = df[bool_cols].astype(int)
        X = df.drop('Churn', axis=1)
        _, X_test, _, _ = train_test_split(
            X, df['Churn'], test_size=0.2, random_state=42, stratify=df['Churn']
        )
        preds = model.predict_proba(X_test)
        assert preds.shape[0] == len(X_test)
        assert preds.shape[1] == 2

    def test_probabilities_between_0_and_1(self):
        model = joblib.load('../outputs/model.pkl')
        df = pd.read_csv('../data/processed/features.csv')
        bool_cols = df.select_dtypes(include='bool').columns
        df[bool_cols] = df[bool_cols].astype(int)
        X = df.drop('Churn', axis=1)
        _, X_test, _, _ = train_test_split(
            X, df['Churn'], test_size=0.2, random_state=42, stratify=df['Churn']
        )
        probs = model.predict_proba(X_test)[:, 1]
        assert probs.min() >= 0.0
        assert probs.max() <= 1.0

    def test_predictions_are_binary(self):
        model = joblib.load('../outputs/model.pkl')
        df = pd.read_csv('../data/processed/features.csv')
        bool_cols = df.select_dtypes(include='bool').columns
        df[bool_cols] = df[bool_cols].astype(int)
        X = df.drop('Churn', axis=1)
        _, X_test, _, _ = train_test_split(
            X, df['Churn'], test_size=0.2, random_state=42, stratify=df['Churn']
        )
        preds = model.predict(X_test)
        assert set(preds).issubset({0, 1})


# ---------------------------------------------------
# Run All Tests
# ---------------------------------------------------

loader = unittest.TestLoader()
suite = unittest.TestSuite()

suite.addTests(loader.loadTestsFromTestCase(TestTenureGroup))
suite.addTests(loader.loadTestsFromTestCase(TestTotalChargesFix))
suite.addTests(loader.loadTestsFromTestCase(TestSupportScore))
suite.addTests(loader.loadTestsFromTestCase(TestChurnEncoding))
suite.addTests(loader.loadTestsFromTestCase(TestModelOutput))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print("\n" + "="*50)
print(f"Tests run:    {result.testsRun}")
print(f"Failures:     {len(result.failures)}")
print(f"Errors:       {len(result.errors)}")
print(f"Status:       {'PASSED' if result.wasSuccessful() else 'FAILED'}")
print("="*50)

test_boundary_zero (__main__.TestTenureGroup) ... ok
test_developing_customer (__main__.TestTenureGroup) ... ok
test_established_customer (__main__.TestTenureGroup) ... ok
test_loyal_customer (__main__.TestTenureGroup) ... ok
test_new_customer (__main__.TestTenureGroup) ... ok
test_no_nulls_after_fix (__main__.TestTotalChargesFix) ... ok
test_space_replaced_by_monthly (__main__.TestTotalChargesFix) ... ok
test_valid_values_unchanged (__main__.TestTotalChargesFix) ... ok
test_all_services_active (__main__.TestSupportScore) ... ok
test_no_services (__main__.TestSupportScore) ... ok
test_partial_services (__main__.TestSupportScore) ... ok
test_score_range (__main__.TestSupportScore) ... ok
test_no_maps_to_zero (__main__.TestChurnEncoding) ... ok
test_sum_correct (__main__.TestChurnEncoding) ... ok
test_yes_maps_to_one (__main__.TestChurnEncoding) ... ok
test_model_loads (__main__.TestModelOutput) ... ok
test_prediction_shape (__main__.TestModelOutput) ... ok
test_predictions_are_binary (_


Tests run:    19
Failures:     0
Errors:       0
Status:       PASSED
